# Nemotron LoRA — train on Kaggle (free 2xT4, 4-bit QLoRA)

**Run HEADLESS:** Save Version -> **Save & Run All (Commit)** (survives disconnects).

**Add Input:** the **competition data** + the model **`nemotron-3-nano-30b-a3b-bf16`** (publisher `metric`). **Accelerator: GPU T4 x2. Internet: On.**

No mamba CUDA kernels needed: the model has a pure-PyTorch fallback. This avoids all the mamba/torch ABI matching (which is fragile). 4-bit NF4 + bf16 fits 2xT4.

## 1. Code + dependencies (use Kaggle's own torch — no reinstall)

In [ ]:
!git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops

## 2. Ensure NO mamba kernels (so the model uses the torch fallback)
A broken/mismatched mamba_ssm crashes the import; a *missing* one falls back cleanly.

In [ ]:
!pip uninstall -y -q mamba-ssm causal-conv1d 2>/dev/null
print("mamba kernels absent -> model will use its pure-PyTorch torch_forward path")

## 3. Competition data (recursive find from the attached dataset)

In [ ]:
import glob, os, shutil
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, "train.csv not found — Add Input -> the competition"
shutil.copy(hits[0], 'data/train.csv'); print('train.csv <-', hits[0])

## 4. EDA + build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train (4-bit QLoRA on 2xT4, torch-fallback Mamba)
Loads the base from the attached model mount (no 60 GB download). The naive SSM path is slower, so we do **1 epoch** + seq 768 for a first finish within the 12 h limit; bump `NUM_EPOCHS` later. A smoke test runs first.

In [ ]:
import os
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '14GiB'
os.environ['SFT_MAX_SEQ_LENGTH'] = '768'
os.environ['NUM_EPOCHS'] = '1'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir /kaggle/working/lora_adapter

## 6. Package the submission

In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip